[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# The GeoDataFrame


## What you will be able to do

Build a GeoDataFrame from coordinates or from text, choose which of several geometry columns is
active, and predict which pandas operations keep a GeoDataFrame and which return a plain DataFrame.

## The idea

### The problem

A geometry column arrives in a table in more than one way. The station register in the **Why
GeoPandas** notebook had a latitude and a longitude, and `points_from_xy` made the points. A table
exported from a database often holds its shapes as text instead, a column of strings such as
`POINT (10.75 59.91)`, and a table that has been through a spreadsheet has lost its geometry
altogether and holds only an id to match on.

Once the geometry is in place, the table goes through ordinary pandas work: filtering rows, choosing
columns, stacking two tables, merging in another table's columns. Most of those steps keep a
GeoDataFrame. A few return a plain DataFrame, which has no spatial methods, and the next spatial step
then fails, sometimes several steps later than the one that dropped them.

### A formal definition

> A GeoDataFrame has one **active geometry column**, the one its spatial methods use, and may hold
> other columns of geometry beside it. The active column is a **GeoSeries**, a pandas Series of
> Shapely geometries that carries the CRS. `set_geometry` makes another column the active one, and
> `rename_geometry` renames it. Selecting and stacking keep the GeoDataFrame type when the result
> still holds the active geometry column. `merge` returns the type of its left table.

### Why it works that way

GeoPandas builds on pandas by subclassing, so each pandas operation decides what kind of object it
returns. Selecting rows, sorting, `head` and `concat` keep every column, the active geometry
included, so they return a GeoDataFrame. Choosing columns that leave the geometry out returns a
DataFrame, because there is nothing spatial left. `merge` returns the type of the table on its left,
so a plain table merged with a GeoDataFrame on the right gives a DataFrame, although its geometry
column keeps its geometries and their CRS. `pd.concat` stacks GeoDataFrames only when they share a CRS, because one geometry column
holds coordinates of one kind, and it raises an error rather than mix degrees with meters.

The geometry column also has to hold geometry objects. A column of WKT strings looks like shapes when
printed, but its values are text until `GeoSeries.from_wkt` parses them.

### Where this shows up

Every notebook in this guide starts from a GeoDataFrame built one of these ways. The **Reading and
Writing Files** notebook writes WKT to a CSV and reads it back, the **Coordinate Reference Systems**
notebook looks at the CRS a GeoSeries carries, and the **Pandas, Deep Dive** guide covers the
operations themselves.

### What this notebook covers

- the geometry column, its type and its coordinates
- a GeoDataFrame from a column of WKT text
- two geometry columns in one table, and switching between them
- what selecting, concatenating and merging keep
- geometry written back out as text
- three mistakes made in building or combining GeoDataFrames

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import geopandas as gpd
import pandas as pd

text = pd.DataFrame({"name": ["Oslo", "Bergen"], "wkt": ["POINT (10.75 59.91)", "POINT (5.32 60.39)"]})
places = gpd.GeoDataFrame(text, geometry=gpd.GeoSeries.from_wkt(text["wkt"]), crs=4326)
print(type(places).__name__, places.crs, places.geometry.name)
print(type(places[["name", "wkt"]]).__name__)
```

```
GeoDataFrame EPSG:4326 geometry
DataFrame
```

The WKT column is parsed into points, which become the geometry column. Leaving that column out of a
selection returns a plain DataFrame.

## Setup

Two libraries for the tables, and GeoPandas and Shapely for the geometry.

- `Path` finds the guide's copy of the country outlines, when the notebook runs beside it
- `numpy` and `pandas` build the made-up register
- `geopandas` makes and combines GeoDataFrames, and the cell prints its version
- `shapely` builds the register's points and the districts

`countries` holds the outlines of Denmark, Finland, Norway, Russia and Sweden. Made with Natural
Earth. The outlines come from Natural Earth's 1:50m map of the world's countries, which is in the
public domain. `weather_stations` is the five named stations at their real positions. `stations` is a
GeoDataFrame of the made-up register of 300 stations, in latitude and longitude, and `districts` is the
eight made-up forecast districts, in meters. Both are drawn from fixed numbers, so they are the same on
every run.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


districts = forecast_districts()
print("geopandas", gpd.__version__, "|", len(stations), "made-up stations,", len(districts),
      "made-up districts,", len(countries), "countries")

geopandas 1.2.0 | 300 made-up stations, 8 made-up districts, 5 countries


## Worked examples

### The geometry column

In [2]:
print(stations.dtypes)
print("active geometry:", stations.geometry.name, "|", type(stations.geometry).__name__)

station          object
latitude        float64
longitude       float64
elevation_m       int64
owner            object
installed         int64
geometry       geometry
dtype: object
active geometry: geometry | GeoSeries


Every column but one has an ordinary pandas type. `geometry` has the type `geometry`, and
`stations.geometry` returns it as a GeoSeries, whichever name the column has.

A GeoSeries of points gives its coordinates back as numbers.

In [3]:
print("x:", stations.geometry.x.head(3).tolist())
print("y:", stations.geometry.y.head(3).tolist())
print("total bounds:", stations.total_bounds.round(2))

x: [10.6233, 13.5681, 24.4802]
y: [61.1793, 64.8072, 70.7203]
total bounds: [ 5.05 58.35 30.58 71.03]


`x` is the longitude and `y` the latitude, the same numbers as the register's columns. `total_bounds`
is the rectangle around every station, `(min_x, min_y, max_x, max_y)`, here from 5.05 to 30.58 degrees
east and from 58.35 to 71.03 degrees north.

### A GeoDataFrame from text

A table exported from a database often holds its geometry as WKT. Here the five named stations are
written that way.

In [4]:
as_text = pd.DataFrame({
    "name": weather_stations["name"],
    "wkt": [f"POINT ({lon} {lat})" for lon, lat in zip(weather_stations["longitude"], weather_stations["latitude"])],
})
print(as_text.dtypes)
as_text.head(2)

name    object
wkt     object
dtype: object


,name,wkt
0,Bergen,POINT (5.32 60.39)
1,Oslo,POINT (10.75 59.91)


The `wkt` column is text, of type `object`. `GeoSeries.from_wkt` parses each string into a geometry.

In [5]:
named = gpd.GeoDataFrame(as_text, geometry=gpd.GeoSeries.from_wkt(as_text["wkt"]), crs=4326)
print(type(named).__name__, "| CRS:", named.crs, "| active geometry:", named.geometry.name)
print(list(named.columns))
print(named.geom_type.tolist())

GeoDataFrame | CRS: EPSG:4326 | active geometry: geometry
['name', 'wkt', 'geometry']
['Point', 'Point', 'Point', 'Point', 'Point']


The parsed points become a new column named `geometry`, and the text column stays beside it. Text
carries no CRS, so `crs=4326` says the numbers are degrees.

### A second geometry column in the same table

A district's `representative_point` is a point guaranteed to lie inside it, which is a good place for
a label. Stored as a second column, it sits beside the polygons.

In [6]:
labeled = districts.copy()
labeled["label_point"] = labeled.representative_point()
print(labeled.dtypes)
print("active geometry:", labeled.geometry.name)
print("every label point inside its district:", labeled["label_point"].within(labeled.geometry).all())

district         object
geometry       geometry
label_point    geometry
dtype: object
active geometry: geometry
every label point inside its district: True


Both columns have the type `geometry`, and the polygons stay active. `set_geometry` switches to the
points, and every spatial method then uses them.

In [7]:
as_points = labeled.set_geometry("label_point")
print("active geometry:", as_points.geometry.name, "|", as_points.geom_type.unique())
print("area of the points:", as_points.area.sum())
print("area of the polygons, km²:", round(labeled.area.sum() / 1e6))

active geometry: label_point | ['Point']
area of the points: 0.0
area of the polygons, km²: 386897


The same table answers with an area of 0 once the points are active, and 386,897 km² while the
polygons are. `rename_geometry` gives the active column a new name and keeps it active.

In [8]:
renamed = districts.rename_geometry("outline")
print(list(renamed.columns), "| active geometry:", renamed.geometry.name)

['district', 'outline'] | active geometry: outline


### What selecting keeps

In [9]:
print("rows by a condition:     ", type(stations[stations["owner"] == "airport"]).__name__)
print("sorted, first five:      ", type(stations.sort_values("elevation_m").head()).__name__)
print("columns with geometry:   ", type(stations[["station", "geometry"]]).__name__)
print("columns without geometry:", type(stations[["station", "owner"]]).__name__)
print("the geometry column:     ", type(stations["geometry"]).__name__)
print("another column:          ", type(stations["owner"]).__name__)
print("geometry dropped:        ", type(stations.drop(columns="geometry")).__name__)

rows by a condition:      GeoDataFrame
sorted, first five:       GeoDataFrame
columns with geometry:    GeoDataFrame
columns without geometry: DataFrame
the geometry column:      GeoSeries
another column:           Series
geometry dropped:         DataFrame


Every result that still holds the active geometry column is a GeoDataFrame. A selection of columns
without it, or a table with it dropped, is a DataFrame, and a single column is a GeoSeries or a Series
depending on whether it holds geometry.

### Stacking tables

`pd.concat` stacks two GeoDataFrames that share a CRS into one.

In [10]:
north = stations[stations["latitude"] > 68]
south = stations[stations["latitude"] < 59]
both = pd.concat([north, south])
print(len(north), "+", len(south), "=", len(both), "|", type(both).__name__, "| CRS:", both.crs)

75 + 12 = 87 | GeoDataFrame | CRS: EPSG:4326


### Merging in another table

A plain table describes each owner. Merged onto the stations, the result's type depends on which
table is on the left.

In [11]:
owners = pd.DataFrame({
    "owner": ["met", "road", "airport", "private"],
    "operator": ["weather service", "road authority", "airport operator", "private owner"],
})
plain_left = owners.merge(stations, on="owner")
geo_left = stations.merge(owners, on="owner")
print("plain table on the left:", type(plain_left).__name__, "| geometry column's type:",
      plain_left["geometry"].dtype)
print("GeoDataFrame on the left:", type(geo_left).__name__, "| CRS:", geo_left.crs)
try:
    plain_left.area
except AttributeError as error:
    print("AttributeError:", error)

plain table on the left: DataFrame | geometry column's type: geometry
GeoDataFrame on the left: GeoDataFrame | CRS: EPSG:4326
AttributeError: 'DataFrame' object has no attribute 'area'


With the plain table on the left, the result is a DataFrame, which has no spatial methods, so `area`
raises `AttributeError`. Its geometry column still holds geometries in EPSG:4326, and naming it as the
geometry makes a GeoDataFrame again. With the GeoDataFrame on the left, the result is a GeoDataFrame in
EPSG:4326 from the start.

In [12]:
restored = gpd.GeoDataFrame(plain_left, geometry="geometry")
print(type(restored).__name__, "| CRS:", restored.crs, "|", len(restored), "rows")

GeoDataFrame | CRS: EPSG:4326 | 300 rows


### Geometry back to text

`to_wkt` turns the geometry column into WKT strings, which a CSV file or a spreadsheet can hold.

In [13]:
exported = stations.to_wkt()
print(type(exported).__name__, "|", exported["geometry"].dtype)
print(exported[["station", "geometry"]].head(2))

DataFrame | object
  station                 geometry
0   NS001  POINT (10.6233 61.1793)
1   NS002  POINT (13.5681 64.8072)


The result is a plain DataFrame whose `geometry` column holds text, ready to save. Reading it back
needs `GeoSeries.from_wkt` and the CRS, as the WKT example above did.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/03-the-geodataframe-solutions.ipynb).

**1.** Write the five named stations as WKT strings with an f-string, as the worked examples did, but with latitude and longitude rounded to one decimal place, and make a GeoDataFrame from them. Print its CRS and its geometry as WKT.

In [14]:
# your code here

**2.** Keep the stations higher than 1,000 m, and print the type of the result and how many stations it holds.

In [15]:
# your code here

**3.** Make a copy of `districts` with a column `center` holding each district's centroid, make it the active geometry, and print each district's name with the x and y of its center in kilometers, rounded to a whole number.

In [16]:
# your code here

**4.** Merge `owners` onto `stations` so that the result is a GeoDataFrame, and print its type, its CRS and the `station`, `owner` and `operator` of its first three rows.

In [17]:
# your code here

**5.** Find Norway in `countries` by its three-letter code, without using its name, and print its `POP_EST`. Print the code columns first, since Natural Earth's codes have surprises.

In [18]:
# your code here

## Common errors

### `TypeError: Input must be valid geometry objects: POINT (5.32 60.39)`

The WKT table's text column is named as the geometry.

In [19]:
text_table = pd.DataFrame({"name": ["Bergen", "Oslo"], "wkt": ["POINT (5.32 60.39)", "POINT (10.75 59.91)"]})
gpd.GeoDataFrame(text_table, geometry="wkt", crs=4326)

TypeError: Input must be valid geometry objects: POINT (5.32 60.39)

`geometry="wkt"` asks GeoPandas to use the column as it is, and the column holds strings. The message
quotes the first value that is not a geometry. Parse the text first.

In [20]:
fixed = gpd.GeoDataFrame(text_table, geometry=gpd.GeoSeries.from_wkt(text_table["wkt"]), crs=4326)
print(fixed.geom_type.tolist())

['Point', 'Point']


### No error, and no rows for Norway

In [21]:
print(len(countries[countries["ISO_A3"] == "NOR"]), "rows")
print(countries[["ADMIN", "ADM0_A3", "ISO_A3"]])

0 rows
     ADMIN ADM0_A3 ISO_A3
0  Denmark     DNK    DNK
1  Finland     FIN    FIN
2   Norway     NOR    -99
3   Russia     RUS    RUS
4   Sweden     SWE    SWE


Natural Earth gives Norway an `ISO_A3` code of `-99`, so filtering for `"NOR"` returns an empty table
with no error. `ADM0_A3` holds `NOR`. Before filtering a real dataset on a code, print the column, as
the second line does.

In [22]:
print(len(countries[countries["ADM0_A3"] == "NOR"]), "row")

1 row


### ``ValueError: Cannot determine common CRS for concatenation inputs, got ['WGS 84', 'WGS 84 / Pseudo-Mercator']. Use `to_crs()` to transform geometries to the same CRS before merging.``

In [23]:
first = stations.iloc[:2]
second = stations.iloc[2:4].to_crs(3857)
pd.concat([first, second])

ValueError: Cannot determine common CRS for concatenation inputs, got ['WGS 84', 'WGS 84 / Pseudo-Mercator']. Use `to_crs()` to transform geometries to the same CRS before merging.

The second table's points were converted to Web Mercator, EPSG:3857, whose numbers are meters, and a
single geometry column cannot hold degrees and meters together. Convert one table to the other's CRS
before stacking them.

In [24]:
stacked = pd.concat([first, second.to_crs(first.crs)])
print(len(stacked), "rows | CRS:", stacked.crs)

4 rows | CRS: EPSG:4326


## Recap

- You can make a GeoDataFrame from coordinates with `points_from_xy` or from text with
  `GeoSeries.from_wkt`, and give it its CRS.
- You can keep several geometry columns and choose the active one with `set_geometry`.
- You can tell which selections keep a GeoDataFrame and which return a DataFrame.
- You can merge with the GeoDataFrame on the left, or make a merged DataFrame a GeoDataFrame again
  with `geometry=`.
- You can check a real dataset's codes before filtering on them.

## What is next

The **Coordinate Reference Systems** notebook looks at the CRS every GeoSeries carries, and at
converting geometries from degrees to meters.

---

&#8592; **Previous:** [Shapely Geometries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/02-shapely-geometries.ipynb)  &nbsp;·&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Coordinate Reference Systems](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/04-coordinate-reference-systems.ipynb) &#8594;
